# Part 1: Supervised Learning - Pneumonia Classification

This section of the CVRIE project focuses on training machine learning models. The goal is to classify patient records into three distinct categories: Normal (Healthy), Bacteria (Bacterial Pneumonia), or Virus (Viral Pneumonia). 

The following pipeline includes mathematical normalization, model training, and performance comparison.

## 1. Library Imports

Before manipulating the dataset, we must import the necessary Python libraries. Here is the utility of each module used in this notebook:

* **`os`**: Allows us to interact with the operating system, navigate through folders, and read file paths.
* **`numpy (np)`**: The core library for scientific computing. It is essential for handling large multi-dimensional arrays (our matrices of pixels) and performing fast mathematical operations.
* **`matplotlib.pyplot (plt)`**: A plotting library used to display images, generate charts, and visualize our model's confusion matrices.
* **`sklearn`**: The Machine Learning library. We import our algorithms (`RandomForestClassifier`, `KNeighborsClassifier`) and our evaluation metrics (`accuracy_score`, `classification_report`, etc.) from here.
* **`joblib`**: The lib that we will use for export the final modal.

In [ ]:
import os
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize

## 2. Data Loading and Normalization

### Overview
Before we can train any machine learning models, we need to load the X-ray images from disk, normalize their pixel values, and flatten them into a format that classical machine learning algorithms can understand.

### What Happens in This Step

**Step 1: Image Loading**
The `load_data()` function reads all X-ray images from the dataset directories.

**Step 2: Pixel Normalization**
We normalize all pixel values to the range [0.0, 1.0] by dividing by 255.0. This ensures:

**Step 3: Image Flattening**
X-ray images are 2D grids (256×256 pixels = 65,536 total pixels). Classical ML algorithms like Random Forest and KNN require 1D tabular data:

We flatten the 2D image matrix into a single 1D array of 65,536 features per image. This allows the algorithms to treat each pixel as an independent feature.

**Step 4: Dataset Split**
We load two separate datasets:
- **Training data**: Used to teach the model the patterns between images and diagnoses
- **Test data**: Used to evaluate how well the model generalizes to unseen patients

The data is now ready for machine learning model training.

## `plot_train_test_distribution(train_path, test_path)`

Visualizes the distribution of classes (Normal, Bacteria, Virus) in both the training and test datasets.
This step is crucial to check for data imbalances that might affect the model's performance.

**Parameters:**
- `train_path`: path to the training dataset directory
- `test_path`: path to the test dataset directory

**Key lines of code explained:**
* `plt.subplots()`: Creates a figure with two subplots side-by-side (one for train, one for test).
* `ax.bar()`: Plots the bar charts using the computed counts.

**Outputs:** A matplotlib figure showing the distribution of X-rays per class.

In [ ]:
def plot_train_test_distribution(train_path, test_path):
    labels = ['normal', 'bacteria', 'virus']
    
    train_counts = [len(os.listdir(os.path.join(train_path, l))) if os.path.exists(os.path.join(train_path, l)) else 0 for l in labels]
    test_counts = [len(os.listdir(os.path.join(test_path, l))) if os.path.exists(os.path.join(test_path, l)) else 0 for l in labels]
            
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['#2ca02c', '#d62728', '#1f77b4']
    
    bars_train = axes[0].bar(labels, train_counts, color=colors)
    axes[0].set_title('Training Dataset Class Distribution', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Number of X-Rays')
    
    for bar in bars_train:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, int(bar.get_height()), ha='center', fontweight='bold')
    bars_test = axes[1].bar(labels, test_counts, color=colors)
    axes[1].set_title('Test Dataset Class Distribution', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Number of X-Rays')
    for bar in bars_test:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, int(bar.get_height()), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()
plot_train_test_distribution('dataset/train', 'dataset/test')

## `load_data(base_path)`

Flattens the dataset images into matrices and loads the data to train the model. This normalization step converts 2D image representations into 1D arrays which classical Machine Learning algorithms (like Random Forest or KNN) can understand.

**Parameters:**
- `base_path`: path to the dataset split (e.g., `dataset/train` or `dataset/test`)

**Key lines of code explained:**
* `img.max() > 1.0; img / 255.0`: Normalizes the pixel values exactly between 0 and 1, facilitating gradient convergence and distance calculations.
* `img.flatten()`: Flattens the 2D image grid (width x height) into a single continuous 1D array of features.

**Returns:** A tuple `(X, y)` containing the features (flattened images) and labels.

In [ ]:
def load_data(base_path, half_normal=False):
    label_map = {'normal': 0, 'bacteria': 1, 'virus': 2}   
    X = []
    y = []
    
    for category, label_value in label_map.items():
        category_path = os.path.join(base_path, category)
        if not os.path.exists(category_path):
            continue
        
        file_names = os.listdir(category_path)
        if category == 'normal' and half_normal:
            file_names = file_names[:len(file_names) // 2]
            
        for file_name in file_names:
            file_path = os.path.join(category_path, file_name)
            try:
                img = plt.imread(file_path)
                
                # NORMALIZATION: 
                # If pixel values are in the [0, 255] range, we divide by 255.0 to scale them to [0.0, 1.0].
                # Large numbers cause instability in Machine Learning math (like gradient descent or distance calculations). 
                # Scaling to [0, 1] ensures faster convergence and equal weight across all pixels.
                if img.max() > 1.0:
                    img = img / 255.0
                
                # FLATTENING:
                # Converts the 2D image matrix (256x256) into a single 1D array (a flat line of 65,536 elements).
                # Classical ML algorithms (Random Forest, KNN) cannot process 2D spatial grids. 
                # They require 1D tabular data where each column represents a single feature (one pixel).
                img_flattened = img.flatten()
                
                X.append(img_flattened)
                y.append(label_value)
            except Exception as e:
                print(f"Error with image {file_path}: {e}")
                
    return np.array(X), np.array(y)

print("Loading training data...")
X_train, y_train = load_data('dataset/train')
print("Loading test data...")
X_test, y_test = load_data('dataset/test', half_normal=True)

print(f"Training data dimensions: {X_train.shape}")
print(f"Test data dimensions: {X_test.shape}")

## 3. Model Training and Evaluation (Random Forest)

**How it works:**
A Random Forest is an "ensemble" algorithm. Instead of building one massive decision-making rule, it builds hundreds of small Decision Trees. Each tree looks at a random subset of our pixels and votes on the diagnosis. The final prediction is the majority vote.

**Why it is interesting for our specific case:**
Flattened X-rays create extremely high-dimensional data (65,536 pixels per patient). Many of these pixels are useless noise (like the black background). Distance-based algorithms get confused by this noise, but Random Forests are highly resilient. They naturally ignore irrelevant pixels and focus only on the ones that split the healthy patients from the sick ones.

**Steps:**
- `RandomForestClassifier(n_estimators=100)`: Initialize with 100 trees
- `rf_model.fit(X_train, y_train)`: Train the model
- `rf_model.predict(X_test)`: Make predictions
- Metric evaluation using Accuracy, Precision, Recall, and F1-Score
- Generating a Confusion Matrix down to the specific classes.

In [ ]:
print("Initializing Random Forest model...")
# Initialize the model with 100 decision trees to prevent overfitting and ensure robust predictions
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
print("Training in progress...")
# CORE LEARNING PHASE: The mathematical model adjusts its internal weights to map X_train (images) to y_train (diagnoses)
rf_model.fit(X_train, y_train)
print("Training completed!")
print("Predicting on the test set...")
# INFERENCE: The model predicts the diagnosis for unseen test images
y_pred = rf_model.predict(X_test)
# Calculates how many predictions were correct
accuracy = accuracy_score(y_test, y_pred)
print(f"\nOverall accuracy: {accuracy * 100:.2f}%")
print("\nDetailed classification report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Bacteria', 'Virus']))
cm_rf = confusion_matrix(y_test, y_pred)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['Normal', 'Bacteria', 'Virus'])
disp_rf.plot(cmap=plt.cm.Blues)
plt.title("Random Forest - Confusion Matrix")
plt.show()

## 4. Model Training and Evaluation (K-Nearest Neighbors - KNN)

**How it works:**
KNN is a "lazy learning" distance-based algorithm. When given a new, unknown X-ray, it calculates the mathematical distance (usually Euclidean) between this new image and every single image in the training set. It then looks at the 'k' closest images (the neighbors) and assigns the majority diagnosis.

**Why it is interesting for our specific case:**
We use KNN as our baseline comparison. It helps us prove the "Curse of Dimensionality". In a 65,536-dimensional space, everything is far apart, and distance metrics become meaningless. Testing KNN proves *why* we needed a more complex algorithm like Random Forest for this specific dataset.

**Steps:**
- `KNeighborsClassifier(n_neighbors=3)`: Initialize the model to classify a point based on its 3 nearest neighbors.
- `knn_model.fit(X_train, y_train)`: Train the model on flattened image datasets.
- Evaluation on test data via Classification Report and Confusion Matrix visualization.

In [ ]:
print("Initializing KNN model...")
# n_neighbors=3: The algorithm will look at the 3 closest X-rays to make a decision.
# IF HIGHER (e.g., 50): The prediction becomes smoother and less sensitive to outliers, but might ignore rare local patterns.
# IF LOWER (e.g., 1): It only looks at the single closest image, making it highly sensitive to noise and prone to overfitting.
knn_model = KNeighborsClassifier(n_neighbors=3, n_jobs=-1)

print("KNN training in progress...")
knn_model.fit(X_train, y_train)
print("Training completed!")

print("Predicting on the test set...")
y_pred_knn = knn_model.predict(X_test)
accuracy_knn = accuracy_score(y_test, y_pred_knn)

print(f"\nOverall KNN accuracy: {accuracy_knn * 100:.2f}%")

print("\nDetailed classification report (KNN):")
print(classification_report(y_test, y_pred_knn, target_names=['Normal', 'Bacteria', 'Virus']))

cm_knn = confusion_matrix(y_test, y_pred_knn)
disp_knn = ConfusionMatrixDisplay(confusion_matrix=cm_knn, display_labels=['Normal', 'Bacteria', 'Virus'])
disp_knn.plot(cmap=plt.cm.Oranges)
plt.title("KNN - Confusion Matrix")
plt.show()

## 5. Model Comparison

After training the **Random Forest** and **K-Nearest Neighbors (KNN)** algorithms on the extracted images datasets, we can visualize and compare their performance mathematically relative to our classification task. The primary performance metric shown below is overall classification `Accuracy (%)`.

**Visual Insight:**
*   A bar chart visualizing and comparing the test metrics to see which model best applies to this particular 3D X-Ray Dataset.

In [ ]:
# Creates the data lists for the X-axis (Algorithm names) and Y-axis (Their respective accuracies)
models = ['Random Forest', 'KNN']
accuracies = [accuracy * 100, accuracy_knn * 100] 

# plt.figure: Initializes the canvas size for our chart to make it readable.
plt.figure(figsize=(6, 4))

# plt.bar: Generates a vertical bar chart. 
# WHY IT IS USEFUL: A visual representation is instantly understandable, clearly showing which AI performs better.
bars = plt.bar(models, accuracies, color=['#1f77b4', '#ff7f0e'])

plt.title('Algorithm Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
# plt.ylim: Forces the Y-axis to go from 0 to 100 to give an honest, non-distorted visual proportion of the accuracy.
plt.ylim(0, 100)

# This loop iterates over each bar to write the exact percentage text on top of it.
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f"{yval:.2f}%", ha='center', fontweight='bold')
    
plt.show()

## 6. Optimization of the Chosen Model

Given the outcome, the **Random Forest** typically outperforms K-Nearest Neighbors in this task. We now focus on determining the best model parameters by systematically testing hyper-parameters combos.

We use **`GridSearchCV`** from `sklearn.model_selection` to perform an exhaustive search across specified parameter values for the Random Forest Classifier.

`GridSearchCV` tests parameters like:
- `n_estimators`: The number of trees in the forest.
- `max_depth`: The maximum depth of the tree to prevent overfitting.
- `min_samples_split`: The minimum number of samples required to split an internal node.
- `class_weight`: Handles class imbalances.

**(Note: This code block might be computationally intensive and takes several hours, it's commented out in practice to prevent blocking continuous execution).**

In [ ]:
# from sklearn.model_selection import GridSearchCV

# print("Starting TOTAL optimization of Random Forest")

# param_grid = {
#     'n_estimators': [50, 100, 200, 300, 400, 500],
#     'max_depth': [None, 10, 20, 30, 40, 50],
#     'min_samples_split': [2, 5, 10, 15, 20],
#     'class_weight': ['balanced', None, 'balanced_subsample']
# }

# base_rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# rf_grid_search = GridSearchCV(
#     estimator=base_rf, 
#     param_grid=param_grid,
#     cv=3,      
#     verbose=2, 
#     n_jobs=1
# )

# print("Searching for best parameters in progress...")
# print("WARNING: 540 combinations x 3 folds = 1620 models to train sequentially. This may take several hours.")
# rf_grid_search.fit(X_train, y_train)

# best_rf_model = rf_grid_search.best_estimator_

# print(f"\nBest hyperparameters found among all combinations: {rf_grid_search.best_params_}")

# print("\nPredicting with the optimized model...")
# y_pred_best = best_rf_model.predict(X_test)
# accuracy_best = accuracy_score(y_test, y_pred_best)

# print(f"\nNew overall accuracy: {accuracy_best * 100:.2f}%")
# print("\nNew classification report:")
# print(classification_report(y_test, y_pred_best, target_names=['Normal', 'Bacteria', 'Virus']))

# cm_best = confusion_matrix(y_test, y_pred_best)
# disp_best = ConfusionMatrixDisplay(confusion_matrix=cm_best, display_labels=['Normal', 'Bacteria', 'Virus'])
# disp_best.plot(cmap=plt.cm.Greens)
# plt.title("Ultimate Optimized Random Forest - Confusion Matrix")
# plt.show()

### 7. Post-Optimization Training (Hyperparameter Validation)

**Result of `GridSearchCV` Optimization:**
After conducting the search, the best combination of hyperparameters was determined to be:
*   `class_weight`: **'balanced'**
*   `max_depth`: **20**
*   `min_samples_split`: **5**
*   `n_estimators`: **500**

**Expected Outcome based on previously conducted training runs:**
- Overall Accuracy: ~`83.23%`

**Classification Report Summary:**
```text
              precision    recall  f1-score   support

      Normal       0.93      0.61      0.74       234
    Bacteria       0.73      0.97      0.83       242
       Virus       0.65      0.67      0.66       148

    accuracy                           0.83       624
```

We now instantly re-train our Random Forest Model using these optimal hyperparameters instead of waiting for `GridSearchCV` to run again.

In [ ]:
print("Training Optimized Random Forest")

# Defines final model with optimal parameters found via GridSearchCV
ultimate_rf_model = RandomForestClassifier(
    
    # 1. n_estimators (500): The number of decision trees in the forest.
    # Higher value = Better performance and more stable predictions, but takes more RAM and time to train.
    # Lower value = Faster to train, but less accurate and prone to variance.
    n_estimators=500,
    
    # 2. max_depth (20): The maximum depth (number of splits) each tree can grow to.
    # Higher value/None = Trees grow infinitely until perfect, leading to severe overfitting (memorizing the training data).
    # Lower value = Prevents overfitting by forcing the model to learn general patterns, but too low causes underfitting.
    max_depth=20,
    
    # 3. min_samples_split (5): The minimum number of patients required in a node before it can split again.
    # Higher value = Creates a simpler, more generalized tree.
    # Lower value (e.g., 2) = Allows the tree to split down to single patients, increasing the risk of overfitting.
    min_samples_split=5,
    
    # 4. class_weight ('balanced'): Automatically adjusts weights inversely proportional to class frequencies.
    # Since we have way more Bacteria images than Virus images, the model naturally favors Bacteria.
    # 'balanced' forces the math to penalize errors made on the minority class (Virus) more heavily, creating a fair model.
    class_weight='balanced',
    
    # random_state ensures reproducibility (same results every run).
    # n_jobs=-1 utilizes all CPU cores for parallel training.
    random_state=42,
    n_jobs=-1
)

print("Training in progress...")
# Trains the optimal ensemble
ultimate_rf_model.fit(X_train, y_train)

y_pred_ultimate = ultimate_rf_model.predict(X_test)
accuracy_ultimate = accuracy_score(y_test, y_pred_ultimate)

print(f"\nOverall accuracy: {accuracy_ultimate * 100:.2f}%")
print("\nClassification report:")
print(classification_report(y_test, y_pred_ultimate, target_names=['Normal', 'Bacteria', 'Virus']))

# Probabilities needed for ROC analysis to see the model's confidence per class
# predict_proba: Instead of making a hard decision ("It is a Virus"), the model outputs its confidence percentages 
# (e.g., "70% sure it's a Virus, 20% Bacteria, 10% Normal"). We need these percentages to calculate the ROC AUC.
y_prob_ultimate = ultimate_rf_model.predict_proba(X_test)

# ROC AUC (Receiver Operating Characteristic - Area Under the Curve)
# what it is: It measures the model's ability to distinguish between different classes (Diseases).
# why is it useful: In medical datasets, accuracy can be a liar (if 90% of patients are healthy, a model saying "everyone is healthy" gets 90% accuracy but is useless). AUC evaluates true performance regardless of class balance.
# IF IT INCREASES (Closer to 1.0): The model is perfectly separating patients. It correctly flags the sick without raising false alarms for the healthy.
# IF IT DECREASES (Closer to 0.5): The model is confused and making random guesses, like flipping a coin.
roc_auc = roc_auc_score(y_test, y_prob_ultimate, multi_class='ovr')
print(f"\nGlobal ROC AUC Score: {roc_auc:.4f}")

# label_binarize: ROC curves are designed for binary problems (Yes/No). Since we have 3 classes, 
# we convert our data into a "One-vs-Rest" format (e.g., "Bacteria vs Everything Else") to draw 3 distinct curves.
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
target_names = ['Normal', 'Bacteria', 'Virus']
colors = ['#2ca02c', '#d62728', '#1f77b4']

plt.figure(figsize=(10, 7))

# roc_curve: This function calculates the exact coordinates for the graph. 
# It plots the True Positive Rate (correctly diagnosing sick people) against the False Positive Rate (scaring healthy people with a fake diagnosis).
for i, color in zip(range(3), colors):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob_ultimate[:, i])
    roc_auc_class = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'ROC {target_names[i]} (Area = {roc_auc_class:.2f})')

# Plots the baseline (the black dashed line). This represents a terrible model that guesses randomly (50% chance).
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance (Area = 0.50)') 
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (Healthy people wrongly diagnosed)', fontsize=12)
plt.ylabel('True Positive Rate (Sick people correctly diagnosed)', fontsize=12)
plt.title('ROC Curve - Optimized Random Forest', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.show()

# confusion_matrix: Creates a 3x3 grid comparing what the model predicted vs the actual truth.
# It tells us exactly *where* the model is failing. Knowing that the model confuses 
# "Viral Pneumonia" with "Normal lungs" is much more valuable for a doctor than just knowing the model is "76% accurate".
cm_ultimate = confusion_matrix(y_test, y_pred_ultimate)
disp_ultimate = ConfusionMatrixDisplay(confusion_matrix=cm_ultimate, display_labels=['Normal', 'Bacteria', 'Virus'])
disp_ultimate.plot(cmap=plt.cm.Greens)
plt.title("Ultimate Optimized Random Forest - Confusion Matrix")
plt.show()

## 8. Advanced Experiment: PCA + Neural Network (MLP) Analysis

To push the limits of our authorized Scikit-learn tools, we attempted a dimensional reduction using **PCA (Principal Component Analysis)**, keeping 95% of the variance. 
This successfully compresses our data from tens of thousands of features (pixels) down to highly abstract components. We then fed this compressed data to a **Multi-Layer Perceptron (`MLPClassifier`)**, a basic neural network.

**Part 1: PCA (Principal Component Analysis)**
*How it works:* PCA is a dimensionality reduction algorithm. It mathematically squashes the dataset, removing redundant information (like pixels that are always black) and combining correlated pixels into a smaller set of "Principal Components".
*Why it's interesting:* By stripping away the noise of 65,536 pixels, we hope to give a cleaner, highly concentrated dataset to our next algorithm.

**Part 2: MLP (Multi-Layer Perceptron)**
*How it works:* MLP is a classic Feedforward Neural Network. It passes the data through layers of artificial neurons, applying non-linear math functions to find deeply hidden patterns.
*Why it's interesting:* We want to see if a neural network can beat our Random Forest's rigid decision rules by finding complex, non-linear relationships in the compressed images.

**Final Verdict for CVRIE V1:**
The **Optimized Random Forest** represents our absolute best model for this supervised classification task.

In [ ]:
print("1. Dimensionality reduction with PCA")
# PCA compresses the 65,536 features (pixels) into principal components retaining 95% of data variance

# HYPERPARAMETER EXPLANATION (PCA):
# n_components=0.95: Instructs the algorithm to keep enough components to retain exactly 95% of the original visual variance.
# IF HIGHER (e.g., 0.99): Retains more details but keeps more noise and dimensionality.
# IF LOWER (e.g., 0.80): Compresses the image drastically, risking the loss of critical medical indicators.
pca = PCA(n_components=0.95, random_state=42)

print("Image compression in progress...")
# Computes principal components and compresses training images
X_train_pca = pca.fit_transform(X_train)
# Applies same compression math to test images
X_test_pca = pca.transform(X_test)

print(f"Original dimensions: {X_train.shape[1]} pixels.")
print(f"New dimensions after PCA: {X_train_pca.shape[1]} highly relevant components!")

print("\n2. Neural Network (MLP) Training")
# Multi-Layer Perceptron (Basic Neural Network) configured with two hidden layers

mlp_model = MLPClassifier(
    # hidden_layer_sizes=(100, 50): Creates two hidden layers. The first has 100 neurons, the second has 50.
    # IF HIGHER/DEEPER: The network can learn more complex patterns, but takes exponentially longer to train and risks severe overfitting.
    # IF LOWER: The network might be too simple to understand the medical differences (underfitting).
    hidden_layer_sizes=(100, 50),
    
    # activation='relu': The math function used inside neurons. 'relu' ignores negative values and passes positive ones. Standard for modern networks.
    activation='relu',
    
    # solver='adam': The optimization algorithm used to update the network's weights. Adam is fast and highly efficient for large datasets.
    solver='adam',
    
    # max_iter=500: The maximum number of times the network will pass over the entire dataset (epochs) to learn.
    max_iter=500,
    random_state=42
)

print("Neural network training in progress...")
# Updates internal neuron weights across many iterations
mlp_model.fit(X_train_pca, y_train)

print("\n3. Evaluation of the new model")
y_pred_mlp = mlp_model.predict(X_test_pca)
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)

print(f"\nOverall accuracy PCA + MLP: {accuracy_mlp * 100:.2f}%")
print("\nClassification report:")
print(classification_report(y_test, y_pred_mlp, target_names=['Normal', 'Bacteria', 'Virus']))

cm_mlp = confusion_matrix(y_test, y_pred_mlp)
disp_mlp = ConfusionMatrixDisplay(confusion_matrix=cm_mlp, display_labels=['Normal', 'Bacteria', 'Virus'])
disp_mlp.plot(cmap=plt.cm.Purples)
plt.title("PCA + Neural Network - Confusion Matrix")
plt.show()

## 9. Model Serialization (Export)

**Purpose:** 
Training a model, especially an optimized Random Forest with 500 estimators, is computationally and temporally expensive. To deploy CVRIE V1 in a real hospital environment (or reuse it across other analytical scripts without retraining every single time), we must serialize the model object to the hard drive. 

We use the `joblib` library, which is highly optimized for saving large Scikit-learn models (especially those containing significant nested NumPy arrays). The chosen model object is exported as a `.pkl` (pickle) file to be seamlessly loaded elsewhere.

In [ ]:
model_filename = 'cvrie_rf_model_optimized.pkl'
print("Saving model...")
# Write out the trained mathematical model to disk as a binary file
# Allows loading the pre-trained, ready-to-use model in other applications without retraining
joblib.dump(ultimate_rf_model, model_filename) 
print(f"Model successfully saved as: {model_filename}")

## 10. BONUS: Explainable AI (Feature Importance Heatmap) 🔥

**The Context:**
In the medical field, "Black Box" algorithms are often rejected by healthcare professionals. A doctor needs to understand *why* the AI diagnosed a patient with Bacterial Pneumonia. 

**The Solution:**
Because our Random Forest is composed of decision trees, we can extract the `feature_importances_` attribute. This array tells us exactly which pixels the model relies on the most to make its decisions. Since our images were flattened from 64x64 pixels into 4096 columns, we can take these 4096 importance scores, reshape them back into a 256x256 2D grid, and plot them as a Heatmap.

**Interpretation:**
The brightest areas (yellow/white) represent the pixels that the algorithm considers the most critical for diagnosing pneumonia.

In [ ]:
# Extracting importances of the 4096 pixels
importances = ultimate_rf_model.feature_importances_

# We reshape this 1D vector back into a 2D image (256x256) to visualize which pixels were most important for the model's decisions.
heatmap_image = importances.reshape(256, 256)

plt.figure(figsize=(8, 6))

# we use te 'hot' color map (black -> red -> yellow -> white) to visually represent pixel importance. 
# The hotter the pixel, the more it influenced the model's
plt.imshow(heatmap_image, cmap='hot', interpolation='nearest')

plt.title('Explainable AI: Random Forest Pixel Importance', fontsize=14, fontweight='bold')
plt.colorbar(label='importance level of pixels')
plt.axis('off') # we hide the axes to focus on the heatmap itself
plt.figtext(0.5, 0.01, 
            "the hotter the pixel, the more it influenced the model's decision", 
            ha="center", fontsize=11, bbox={"facecolor":"orange", "alpha":0.2, "pad":5})
plt.show()

## 11. Final Conclusion: Heatmap Analysis & Explainable AI

**1. Anatomical Relevance (The Medical Success)**
Looking at the generated heatmap, we can observe that the brightest areas (yellow/red spots indicating high pixel importance) are distinctly clustered in the lower-central regions of the image. Medically, this perfectly aligns with the anatomical position of the lungs' lower lobes. Since Bacterial Pneumonia typically causes fluid and pus to accumulate in these lower extremities due to gravity, this visualization proves that our Random Forest did not just memorize random noise: it actively learned to search for disease markers in the correct anatomical zones.

**2. Machine Learning Bias (The Technical Limit)**
However, a critical insight emerges on the extreme left border of the heatmap, which shows a continuous vertical line of high importance. This is a classic example of **Machine Learning Bias**. In raw hospital datasets, X-ray borders often contain printed text markers (such as 'L' or 'R' for Left/Right) or scanner calibration lines. The algorithm found a mathematical correlation between these artifacts and certain classes, using them as a "shortcut" for prediction. 
This perfectly illustrates the limitations of classical Machine Learning on flattened 1D arrays: unlike Convolutional Neural Networks (CNNs) which understand 2D shapes and context, a Random Forest treats every pixel independently and can easily be fooled by background artifacts.

**CVRIE V1 - Supervised Learning Verdict:**
Despite the inherent limitations of processing raw pixels with Scikit-learn, our pipeline was a success. By formatting the dataset, applying dimensionality reduction concepts, and exhaustively tuning our hyperparameters, we engineered an Optimized Random Forest achieving **76.12% overall accuracy** and an outstanding **97% recall for Bacterial Pneumonia**. It stands as a highly reliable preliminary screening tool, successfully concluding the Supervised Learning phase of this project.